# Female annotation vs. round7 refractive track consistency

## tl;dr

- The female annotation is **not consistent with one fixed geometry ID**: the best fixed label is `ID_1` at only 43.3% of the 298 mapped confirmed annotations.
- The sparse GT sequence contains **at least 26 observed female-ID changes** across 27 runs.
- Annotation-to-geometry evidence is strong overall: 94.3% two-view agreement and 99.3% consensus mapping.
- The conclusion survives the strict 8 px sensitivity filter: the best fixed ID reaches only 44.2% across 265 high-confidence rows.

## Context & Methods

This diagnostic notebook compares the authoritative human female ledger with the exact round7 refractive `_3d.csv` named by the annotator. It validates keys and audit history, joins the selected geometry ID at each annotated frame, reprojects the selected 3D coordinate into each side view, reproduces the clean 8 mm provisional tracklet segmentation, and counts female geometry-ID runs and transitions.

### Key Assumptions

- Geometry IDs are provisional tracker labels, not biological identities.
- A change between adjacent sparse GT samples is an observed lower bound; additional swaps can occur and reverse between samples.
- The user-supplied GUI command states an 8 px gate, while the saved candidate summary records 80 px. The strict sensitivity subset therefore independently requires two-view agreement plus <=8 px click and reprojection distances.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

cwd = Path.cwd().resolve()
REPORT_DIR = (
    cwd if cwd.name == 'female_sex_gt_consistency'
    else (cwd / 'report/20260716/female_sex_gt_consistency').resolve()
)
sys.path.insert(0, str(REPORT_DIR))
from female_track_consistency_analysis import run_analysis

summary = run_analysis()
summary['verdict']

'geometry_ids_not_biologically_consistent_for_female'

## Data

In [2]:
pd.DataFrame({
    'check': [
        'Annotation rows', 'Confirmed female rows', 'Mapped consensus rows',
        'Current track rows', 'Track fish IDs', 'Track backend',
        'Provenance hash matches candidate build', 'Audit rows match canonical ledger'
    ],
    'value': [
        summary['annotation']['row_count'],
        summary['annotation']['confirmed_female_count'],
        summary['mapping']['mapped_consensus_count'],
        summary['track']['row_count'],
        ', '.join(summary['track']['fish_ids']),
        summary['track']['backend'],
        summary['source_consistency']['provenance_sha256_matches'],
        f"{summary['audit']['canonical_rows_with_latest_audit_match']}/"
        f"{summary['audit']['canonical_row_count']}",
    ]
})

,check,value
0,Annotation rows,301
1,Confirmed female rows,300
2,Mapped consensus rows,298
3,Current track rows,1045800
4,Track fish IDs,"ID_1, ID_2, ID_3"
5,Track backend,refractive
6,Provenance hash matches candidate build,True
7,Audit rows match canonical ledger,301/301


## Results

### No fixed geometry ID represents the female

Counts below use confirmed rows with a non-empty cross-view consensus ID.

In [3]:
pd.read_csv(REPORT_DIR / 'identity_counts.csv')

,geometry_id,annotation_count,share
0,ID_1,129,0.432886
1,ID_2,68,0.228188
2,ID_3,101,0.338926


### The dominant female label changes across the recording

Temporal bins come from the saved candidate metadata; the two manual-jump rows have no bin and are retained separately.

In [4]:
pd.read_csv(REPORT_DIR / 'temporal_identity_counts.csv')

,temporal_bin,geometry_id,annotation_count,share_within_bin
0,early,ID_1,19,0.166667
1,early,ID_2,17,0.149123
2,early,ID_3,78,0.684211
3,middle,ID_1,52,0.553191
4,middle,ID_2,37,0.393617
5,middle,ID_3,5,0.053191
6,late,ID_1,57,0.647727
7,late,ID_2,13,0.147727
8,late,ID_3,18,0.204545
9,NaN,ID_1,1,0.500000


### Strict sensitivity filtering does not rescue a fixed ID

This filter removes one-view rows, disagreements, and every selected pose whose click or selected-ID reprojection exceeds 8 px in either side camera.

In [5]:
pd.DataFrame({
    'subset': ['all mapped', 'strict two-view', 'very strict candidate'],
    'n': [
        summary['mapping']['mapped_consensus_count'],
        summary['sensitivity']['strict_two_view_count'],
        summary['sensitivity']['very_strict_count'],
    ],
    'best_fixed_id_accuracy': [
        summary['identity_consistency']['best_fixed_geometry_id_accuracy'],
        summary['sensitivity']['strict_two_view_best_fixed_accuracy'],
        summary['sensitivity']['very_strict_best_fixed_accuracy'],
    ],
    'observed_switches': [
        summary['identity_consistency']['observed_id_switch_count'],
        summary['sensitivity']['strict_two_view_observed_switch_count'],
        summary['sensitivity']['very_strict_observed_switch_count'],
    ],
})

,subset,n,best_fixed_id_accuracy,observed_switches
0,all mapped,298,0.432886,26
1,strict two-view,265,0.441509,25
2,very strict candidate,249,0.449799,23


### Camera-link evidence is mostly strong, with a small wide-gate tail

In [6]:
pd.read_csv(REPORT_DIR / 'reprojection_summary.csv')

,view,mapped_selection_count,reprojection_median_px,reprojection_p95_px,reprojection_max_px,reprojection_over_8px_count,click_to_pose_median_px,click_to_pose_p95_px,click_to_pose_max_px,click_to_pose_over_8px_count
0,side1,294,1.864692,6.091934,22.311027,11,1.460592,5.512862,12.283229,4
1,side2,291,2.101861,5.493386,74.438425,11,1.254944,4.934905,11.931419,4


### Observed female-ID transition brackets

These brackets localize changes only between consecutive annotated frames; they do not claim the exact swap frame.

In [7]:
pd.read_csv(REPORT_DIR / 'observed_female_id_transitions.csv')

,left_annotation_frame,right_annotation_frame,from_geometry_id,to_geometry_id,geometry_id_consistency,candidate_frame,candidate_max_reprojection_distance_px,gap_frames,left_seconds,right_seconds
0,7438,10198,ID_2,ID_3,agree,10198,2.840593,2760,44.807229,61.433735
1,19436,20234,ID_3,ID_1,one_view,20234,1.894375,798,117.084337,121.891566
2,22165,22966,ID_1,ID_3,agree,22966,3.794349,801,133.524096,138.349398
3,28363,28884,ID_3,ID_2,one_view,28884,4.342394,521,170.861446,174.000000
4,30411,31616,ID_2,ID_3,agree,31616,13.300590,1205,183.198795,190.457831
5,70835,71288,ID_3,ID_1,agree,71288,3.389687,453,426.716867,429.445783
6,71288,72371,ID_1,ID_3,agree,72371,2.146285,1083,429.445783,435.969880
7,72371,73027,ID_3,ID_2,agree,73027,2.123450,656,435.969880,439.921687
8,76698,77285,ID_2,ID_1,agree,77285,4.998344,587,462.036145,465.572289
9,78799,79547,ID_1,ID_3,agree,79547,10.584633,748,474.692771,479.198795


## Takeaways

In [8]:
json.dumps({
    'verdict': summary['verdict'],
    'identity_consistency': summary['identity_consistency'],
    'sensitivity': summary['sensitivity'],
    'warnings': summary['warnings'],
}, indent=2)

'{\n  "verdict": "geometry_ids_not_biologically_consistent_for_female",\n  "identity_consistency": {\n    "geometry_id_counts": [\n      {\n        "geometry_id": "ID_1",\n        "annotation_count": 129,\n        "share": 0.43288590604026844\n      },\n      {\n        "geometry_id": "ID_2",\n        "annotation_count": 68,\n        "share": 0.22818791946308725\n      },\n      {\n        "geometry_id": "ID_3",\n        "annotation_count": 101,\n        "share": 0.3389261744966443\n      }\n    ],\n    "best_fixed_geometry_id": "ID_1",\n    "best_fixed_geometry_id_accuracy": 0.43288590604026844,\n    "best_fixed_geometry_id_error_rate": 0.5671140939597316,\n    "observed_id_switch_count": 26,\n    "observed_run_count": 27,\n    "adjacent_mapped_annotation_pair_count": 297,\n    "observed_switch_rate_per_adjacent_annotation_pair": 0.08754208754208755\n  },\n  "sensitivity": {\n    "strict_two_view_definition": "confirmed F; side views agree; both selected-pose reprojection distances an

The human annotations are suitable as sparse identity anchors after the annotator provenance is corrected. They should not be converted into a single `fish_id -> female` lookup. Use the transition brackets as supervision for identity stitching, and manually review the two cross-view disagreements plus the large side2 reprojection outlier at frame 246665 before freezing a gold-standard release.